In [1]:
from llama_cpp import Llama
import json
import re
import logging
import sqlparse
from datetime import datetime

In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [3]:
llm = Llama(
    model_path=r"C:\Users\HP\Documents\GINF2\StagePfa_INVOLYS\Model\mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    n_ctx=8192,
    n_threads=8,
    verbose=False
)

llama_context: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [4]:
perimetre = {
    "tables": {
        "commandes": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "date_commande": "DATE",
                "montant": "DECIMAL(10,2)",
                "client_id": "INT FOREIGN KEY",
                "produit_id": "INT FOREIGN KEY"
            },
            "description": "Commandes passées par les clients"
        },
        "clients": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "secteur": "VARCHAR(50)",
                "ville": "VARCHAR(50)"
            },
            "description": "Informations clients"
        },
        "produits": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "categorie": "VARCHAR(50)",
                "prix_unitaire": "DECIMAL(10,2)"
            },
            "description": "Catalogue produits"
        },
        "fournisseurs": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "pays": "VARCHAR(50)",
                "categorie_fournisseur": "VARCHAR(50)"
            },
            "description": "Fournisseurs disponibles"
        }
    },
    "relations": {
        "commandes.client_id": "clients.id",
        "commandes.produit_id": "produits.id"
    }
}


In [5]:
def format_schema_detailed(schema):
    tables_desc = []
    for table, info in schema["tables"].items():
        cols = ", ".join([f"{col} ({type_col})" for col, type_col in info["columns"].items()])
        tables_desc.append(f"Table {table}: {cols} - {info['description']}")
    
    relations = "\nRelations:\n" + "\n".join([f"- {rel}" for rel in schema["relations"]])
    
    return "\n".join(tables_desc) + relations


In [6]:
def prompt_analyse_avance(question, schema):
    return f"""
Tu es un moteur d’analyse linguistique expert intégré à un système ERP. Ta tâche est d’analyser la question ci-dessous et d’en extraire toutes les informations nécessaires à la génération d’une requête SQL. 

QUESTION UTILISATEUR :
"{question}"

SCHÉMA DE LA BASE DE DONNÉES :
{format_schema_detailed(schema)}

OBJECTIF :
Analyse la question et produis une sortie **au format texte structuré strict**, contenant toutes les sections obligatoires, même si certaines sont vides. N’invente jamais de table ou colonne non mentionnée dans le schéma.

FORMAT À RESPECTER (Suit la forme exacte ci-dessous) :

INTENTION: ...
TABLES: [...]
COLONNES: [...]
FILTRES: [...] (si ils existent bien sur)
JOINTURES: [...] (si elles existent bien sur )
AGRÉGATION: ... (si elle existe bien sur)

...
RÈGLES À RESPECTER :
- Ne saute **aucune section** même si elle est vide (ex: `FILTRES: []`)
- Écris **les noms de colonnes et tables exactement comme dans le schéma**
- Pour les dates : indique mois et année avec `MONTH = x` et `YEAR = xxxx`
- Pour les comparaisons implicites ("plus de", "moins que", etc.), traduis-les en opérateurs SQL : `>`, `<`, `BETWEEN`, etc.
- Pour les valeurs textuelles, entoure-les avec des quotes simples `'...’`
- Si la question mentionne un client ou produit, garde son nom **exact**

EXEMPLE :
Pour la question : "Quelles sont les commandes passées entre janvier et mars 2024 ?"
La réponse correcte serait :

INTENTION: SELECT  
TABLES: [commandes]  
COLONNES: [commandes.id, commandes.date_commande]  
FILTRES: [EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 3, EXTRACT(YEAR FROM commandes.date_commande) = 2024]  
JOINTURES: []  
AGRÉGATION:

Commence ton analyse maintenant, en respectant **exactement ce format**.


Ta réponse doit suivre CE FORMAT EXACT (même si vide) :

INTENTION: SELECT ou autre chose 
TABLES: [commandes, clients]  
COLONNES: [commandes.id, commandes.date_commande]  
FILTRES: [EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 3, EXTRACT(YEAR FROM commandes.date_commande) = 2024]  
JOINTURES: [commandes.client_id = clients.id]  
AGRÉGATION:  les groupes by....

Commence ton analyse maintenant, en respectant **exactement ce format**.
""".strip()


In [7]:
def prompt_sql_precis(analyse_text, schema):
    return f"""
Tu es un générateur SQL expert, spécialisé dans la transformation d'une analyse sémantique structurée en requête SQL valide et exécutable.

Voici l'analyse complète de la question utilisateur :
{analyse_text}

Voici le schéma de la base de données à respecter scrupuleusement :
{json.dumps(schema, indent=2, ensure_ascii=False)}

OBJECTIF :
Génère une requête SQL **valide**, **optimale** et **lisible**, qui respecte parfaitement les contraintes définies dans l’analyse.

RÈGLES OBLIGATOIRES :
1. N’utilise **que** les colonnes listées dans la section "colonnes"
2. Applique **tous les filtres** listés dans la section "filtres", avec les bons opérateurs SQL
3. Implémente **toutes les jointures** décrites dans "jointures" avec `INNER JOIN`, sauf indication contraire
4. Pour les **dates**, utilise `EXTRACT(MONTH FROM ...)` et `EXTRACT(YEAR FROM ...)` si besoin
5. Pour les **valeurs textuelles**, entoure avec des apostrophes simples : `'exemple'`
6. Si la section "agrégation" contient `group_by`, `order_by`, ou `having`, intègre-les à la requête
7. N’ajoute **aucun alias, colonne ou condition** qui n’est pas mentionnée dans l’analyse

FORMAT DE SORTIE :
Retourne UNIQUEMENT la requête SQL entre les balises suivantes (pas d’explication, pas de commentaire) :

<sql>
-- Requête SQL ici
</sql>
""".strip()


In [8]:
def appeler_llm(prompt, stop=["</sql>"], max_tokens=1024, temperature=0.1):
    try:
        result = llm(
            prompt=prompt, 
            stop=stop, 
            temperature=temperature, 
            max_tokens=max_tokens,
            top_p=0.95,
            repeat_penalty=1.1
        )
        return result["choices"][0]["text"].strip()
    except Exception as e:
        logger.error(f"Erreur LLM: {e}\nPrompt utilisé:\n{prompt}")
        return ""


In [9]:
def nettoyer_sql(code):
    try:
        # Supprimer les balises <sql> et </sql>
        code = re.sub(r'</?sql>', '', code).strip()

        # Formatter proprement
        formatted = sqlparse.format(code, reindent=True, keyword_case='upper')

        # Liste des mots-clés SQL à valider
        keywords = ['SELECT', 'INSERT', 'UPDATE', 'DELETE']
        found = [kw for kw in keywords if kw in formatted.upper()]
        
        if not found:
            logger.warning("⚠️ Aucune instruction SQL reconnue (SELECT, INSERT, etc.)")

        return formatted
    except Exception as e:
        logger.error(f"Erreur nettoyage SQL: {e}")
        return code


In [10]:
def valider_analyse(analyse_text):
    if not analyse_text.strip():
        logger.warning("❌ L’analyse NLP est vide.")
        return False

    # Analyse ligne par ligne
    lignes = analyse_text.strip().splitlines()
    sections_valides = ['INTENTION:', 'TABLES:', 'COLONNES:', 'FILTRES:', 'JOINTURES:', 'AGRÉGATION:']
    sections_trouvees = [ligne.strip().split(":")[0].upper() + ":" for ligne in lignes if ":" in ligne]

    for section in sections_valides:
        if section not in sections_trouvees:
            logger.warning(f"⚠️ Section manquante ou mal formatée : {section}")
            return False

    return True


In [11]:
def traiter_question(question):
    logger.info(f"🟢 Requête reçue : {question}")
    result = {"analyse": None, "sql": None, "erreur": None}

    # Étape 1: Analyse NLP
    try:
        analyse_prompt = prompt_analyse_avance(question, perimetre)
        analyse_text = appeler_llm(analyse_prompt, stop=[], max_tokens=512)

        print("\n🧠 ANALYSE GÉNÉRÉE PAR LE MODÈLE :\n")
        print(analyse_text)
        print("\n" + "="*60 + "\n")

        if not valider_analyse(analyse_text):
            raise ValueError("Analyse incomplète ou mal structurée")

        logger.info("✅ Analyse NLP réussie")
        result["analyse"] = analyse_text

    except Exception as e:
        logger.error(f"❌ Erreur dans l'analyse : {e}")
        result["erreur"] = str(e)
        return result

    # Étape 2: Génération SQL
    try:
        sql_prompt = prompt_sql_precis(analyse_text, perimetre)
        sql_output = appeler_llm(sql_prompt, stop=["</sql>", "\n\n"], max_tokens=512)

        match = re.search(r"<sql>(.*?)(?:</sql>|$)", sql_output, re.DOTALL)
        sql_code = match.group(1).strip() if match else sql_output.strip()

        final_sql = nettoyer_sql(sql_code)

        result["sql"] = final_sql
        logger.info("✅ Requête SQL générée avec succès")

        if "SELECT" not in final_sql.upper():
            logger.warning("⚠️ Requête potentiellement incorrecte")

    except Exception as e:
        logger.error(f"❌ Erreur lors de la génération SQL : {e}")
        result["erreur"] = str(e)

    return result


In [12]:
def tester_exemples_complet():
    """Test avec des cas variés pour valider la robustesse du moteur"""
    exemples = [
        "Quelles sont les commandes passées par le client X en septembre 2024",
        "Combien de commandes ont été passées par des clients du secteur 'Technology'",
        "Quel est le montant total des commandes pour le produit 'Ordinateur'",
        "Quels sont les clients de la ville 'Paris' qui ont passé des commandes",
        "Quelles sont les commandes de plus de 1000 euros",
        "Liste des clients ayant commandé en 2023",
        "Montre-moi les commandes entre janvier et mars 2024",
        "Quels produits ont été commandés par des clients du secteur 'Finance'",
        "Quel est le nombre de commandes pour chaque client",
        "Affiche les montants moyens des commandes par mois en 2024",
        "Quels sont les clients qui n'ont pas passé de commande en 2024",
        "Donne-moi les commandes du produit 'Imprimante' en avril 2023",
        "Combien de commandes ont été passées par client en 2023",
        "Quelles sont les commandes faites par le client Involys en mai",
        "Montre les commandes dont le montant est entre 500 et 2000 euros",
        "Liste les produits commandés plus de 10 fois",
        "Quels sont les clients ayant commandé le produit 'Serveur' en 2022",
        "Quel est le chiffre d'affaires total par secteur en 2024",
        "Quelles commandes ont été faites en dehors de la France",
        "Combien de clients ont passé au moins une commande en 2023",
        "Quels produits de la catégorie 'Logiciel' ont été vendus",
        "Donne la liste des commandes groupées par mois et par produit",
        "Montre les 5 clients qui ont le plus commandé en 2024",
        "Quels sont les produits les plus commandés dans le secteur 'Éducation'",
    ]

    for exemple in exemples:
        print(f"\n{'=' * 80}")
        print(f"TEST: {exemple}")
        print("=" * 80)
        traiter_question(exemple)


In [13]:
def tester_exemples():
    """Test avec des exemples prédéfinis"""
    exemples = [
        "Quelles sont les commandes  passées entre janvier et mars 2024"
    ]
    
    for exemple in exemples:
        print(f"\n{'='*60}")
        print(f"TEST: {exemple}")
        print('='*60)
        traiter_question(exemple)

In [14]:
# === Interface utilisateur améliorée
def main():
    print("🧠 ERP SQL Chatbot Amélioré")
    print("Commandes disponibles:")
    print("  - 'exit' ou 'quit' : Quitter")
    print("  - 'test' : Lancer les exemples de test")
    print("  - 'schema' : Afficher le schéma de la base")
    print("-" * 50)
    
    while True:
        question = input("\n💬 Votre requête : ").strip()
        
        if question.lower() in ["exit", "quit"]:
            print("👋 Au revoir !")
            break
            
        elif question.lower() == "test":
            tester_exemples()
            
        elif question.lower() == "schema":
            print("\n📊 SCHÉMA DE LA BASE:")
            print(format_schema_detailed(perimetre))
            
        elif question:
            traiter_question(question)
        else:
            print("❓ Veuillez saisir une requête valide")

if __name__ == "__main__":
    main()

🧠 ERP SQL Chatbot Amélioré
Commandes disponibles:
  - 'exit' ou 'quit' : Quitter
  - 'test' : Lancer les exemples de test
  - 'schema' : Afficher le schéma de la base
--------------------------------------------------


2025-07-06 20:32:21,392 - INFO - 🟢 Requête reçue : Quelles sont les commandes  passées entre janvier et mars 2024



TEST: Quelles sont les commandes  passées entre janvier et mars 2024


2025-07-06 20:33:19,985 - WARNING - ❌ L’analyse NLP est vide.
2025-07-06 20:33:19,991 - ERROR - ❌ Erreur dans l'analyse : Analyse incomplète ou mal structurée



🧠 ANALYSE GÉNÉRÉE PAR LE MODÈLE :




👋 Au revoir !


In [2]:
from llama_cpp import Llama
import json

# === Schéma simulé
schema = {
    "tables": {
        "commandes": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "date_commande": "DATE",
                "montant": "DECIMAL(10,2)",
                "client_id": "INT FOREIGN KEY",
                "produit_id": "INT FOREIGN KEY"
            },
            "description": "Commandes passées par les clients"
        },
        "clients": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "secteur": "VARCHAR(50)",
                "ville": "VARCHAR(50)"
            },
            "description": "Informations clients"
        },
        "produits": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "categorie": "VARCHAR(50)",
                "prix_unitaire": "DECIMAL(10,2)"
            },
            "description": "Catalogue produits"
        },
        "fournisseurs": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "pays": "VARCHAR(50)",
                "categorie_fournisseur": "VARCHAR(50)"
            },
            "description": "Fournisseurs disponibles"
        }
    },
    "relations": {
        "commandes.client_id": "clients.id",
        "commandes.produit_id": "produits.id"
    }
}

# === Formateur texte pour affichage clair du schéma
def format_schema_detailed(schema):
    tables_desc = []
    for table, info in schema["tables"].items():
        cols = ", ".join([f"{col} ({type_col})" for col, type_col in info["columns"].items()])
        tables_desc.append(f"Table {table}: {cols} - {info['description']}")
    
    relations = "\nRelations:\n" + "\n".join([f"- {rel}" for rel in schema["relations"]])
    return "\n".join(tables_desc) + relations

# === Chargement du modèle
llm = Llama(
    model_path=r"C:\Users\HP\Documents\GINF2\StagePfa_INVOLYS\Model\mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    n_ctx=4096,
    n_threads=8,
    verbose=True
)

# === Prompt à tester
prompt = f"""
Tu es un moteur d’analyse linguistique expert intégré à un système ERP. Ta tâche est d’analyser la question ci-dessous et d’en extraire toutes les informations nécessaires à la génération d’une requête SQL. 

QUESTION UTILISATEUR :
"Quelles sont les commandes  passées entre janvier et mars 2024"

SCHÉMA DE LA BASE DE DONNÉES :
{format_schema_detailed(schema)}

OBJECTIF :
Analyse la question et produis une sortie **au format texte structuré strict**, contenant toutes les sections obligatoires, même si certaines sont vides. N’invente jamais de table ou colonne non mentionnée dans le schéma.

FORMAT À RESPECTER (Suit la forme exacte ci-dessous) :

INTENTION: ...
TABLES: [...]
COLONNES: [...]
FILTRES: [...] (si ils existent bien sur)
JOINTURES: [...] (si elles existent bien sur )
AGRÉGATION: ... (si elle existe bien sur)

RÈGLES À RESPECTER :
- Ne saute **aucune section** même si elle est vide (ex: `FILTRES: []`)
- Écris **les noms de colonnes et tables exactement comme dans le schéma**
- Pour les dates : indique mois et année avec `MONTH = x` et `YEAR = xxxx`
- Pour les comparaisons implicites ("plus de", "moins que", etc.), traduis-les en opérateurs SQL : `>`, `<`, `BETWEEN`, etc.
- Pour les valeurs textuelles, entoure-les avec des quotes simples `'...’`
- Si la question mentionne un client ou produit, garde son nom **exact**

EXEMPLE :
Pour la question : "Quelles sont les commandes passées entre janvier et mars 2024 ?"
La réponse correcte serait :

INTENTION: SELECT  
TABLES: [commandes]  
COLONNES: [commandes.id, commandes.date_commande]  
FILTRES: [EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 3, EXTRACT(YEAR FROM commandes.date_commande) = 2024]  
JOINTURES: []  
AGRÉGATION:

Commence ton analyse maintenant, en respectant **exactement ce format** et utlise bien ton analyse et ton reflexion n'ajoute pas trop de phrase juste ce qui est demandé.
"""

# === Appel du modèle
output = llm(prompt, max_tokens=150)
print("\n🧠 RÉPONSE DU MODÈLE :\n")
print(output["choices"][0]["text"])


llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from C:\Users\HP\Documents\GINF2\StagePfa_INVOLYS\Model\mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama

llama_model_loader: - kv  13:                      tokenizer.ggml.tokens arr[str,32000]   = ["<unk>", "<s>", "</s>", "<0x00>", "<...
llama_model_loader: - kv  14:                      tokenizer.ggml.scores arr[f32,32000]   = [0.000000, 0.000000, 0.000000, 0.0000...
llama_model_loader: - kv  15:                  tokenizer.ggml.token_type arr[i32,32000]   = [2, 3, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...
llama_model_loader: - kv  16:                tokenizer.ggml.bos_token_id u32              = 1
llama_model_loader: - kv  17:                tokenizer.ggml.eos_token_id u32              = 2
llama_model_loader: - kv  18:            tokenizer.ggml.unknown_token_id u32              = 0
llama_model_loader: - kv  19:               general.quantization_version u32              = 2
llama_model_loader: - type  f32:   65 tensors
llama_model_loader: - type q4_K:  193 tensors
llama_model_loader: - type q6_K:   33 tensors
print_info: file format = GGUF V2
print_info: file type   = Q4_K - Medium
print_info: f


🧠 RÉPONSE DU MODÈLE :


ANALYSE:

INTENTION: SELECT
TABLES: [commandes]
COLONNES: [commandes.id, commandes.date_commande]
FILTRES: [commandes.date_commande BETWEEN '2024-01-01' AND '2024-03-31']
JOINTURES: []
AGRÉGATION:

ANALYSE EXPLICITE

La question demande de récupérer les informations sur les commandes passées entre janvier et mars 2024.

On utilise un moteur d’analy
